In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib  # For saving the model

In [ ]:

# Load TSV file
df = pd.read_csv("/kaggle/input/datasets/mrsikandarali/metahate-dataset/your_dataset_updated.tsv", sep='\t')

# Quick check
print(df.head())
print(df.info())



In [ ]:

# 2. Define features and target
# Suppose your text column is named 'text' and target column is 'label'
X = df['text']
y = df['label']

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words='english'
)
vectorizer.fit(X_train)

In [ ]:
from sklearn.linear_model import SGDClassifier
# -----------------------------
# 4. Initialize SGDClassifier (Logistic Regression)
# -----------------------------
clf = SGDClassifier(
    loss='log',         # Logistic Regression
    penalty='l2',
    alpha=1e-4,         # Regularization strength
    max_iter=1,         # We'll manually iterate over epochs
    warm_start=True,    # Keep model between calls to fit()
    n_jobs=-1,
    random_state=42
)

In [ ]:
import time
clf = SGDClassifier(loss='log_loss', penalty='l2', max_iter=1, warm_start=True, n_jobs=-1)
# -----------------------------
# 5. Training with mini-batches and progress info
# -----------------------------
batch_size = 5000  # Adjust depending on memory
epochs = 5         # Number of full passes over data
num_samples = X_train.shape[0]

print("Starting training...")

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    start_time_epoch = time.time()
    
    for start in range(0, num_samples, batch_size):
        end = min(start + batch_size, num_samples)
        
        X_batch = vectorizer.transform(X_train.iloc[start:end])
        y_batch = y_train.iloc[start:end]
        
        clf.partial_fit(X_batch, y_batch, classes=[0,1])
        
        print(f" Trained on samples {start+1} to {end} / {num_samples}")
    
    epoch_time = time.time() - start_time_epoch
    print(f" Epoch {epoch+1} completed in {epoch_time:.2f} seconds")



In [ ]:
# -----------------------------
# 6. Evaluate on test set
# -----------------------------
X_test_tfidf = vectorizer.transform(X_test)
y_pred = clf.predict(X_test_tfidf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Random Forest

In [ ]:
# -----------------------------
# 1. Load dataset
# -----------------------------
# Load TSV file
df = pd.read_csv("/kaggle/input/datasets/mrsikandarali/metahate-dataset/your_dataset_updated.tsv", sep='\t')

X = df['text']
y = df['label']

In [ ]:
import time
# -----------------------------
# 4. Initialize Random Forest with warm_start
# -----------------------------
rf = RandomForestClassifier(
    n_estimators=10,  # Start with 10 trees
    max_depth=None,
    max_features='sqrt',
    n_jobs=-1,
    warm_start=True,
    random_state=42
)

In [ ]:
# -----------------------------
# 2. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# 3. TF-IDF Vectorization
# -----------------------------
vectorizer = TfidfVectorizer(
    max_features=30000,  # Keep smaller for Random Forest
    ngram_range=(1,2),
    stop_words='english'
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf= vectorizer.transform(X_test)

In [ ]:
# -----------------------------
# 5. Train Random Forest incrementally
# -----------------------------
total_trees = 100   # Total trees you want
trees_per_step = 10 # Add 10 trees at a time

print("Starting Random Forest training...")

start_time_total = time.time()

while rf.n_estimators <= total_trees:
    start_time_step = time.time()
    
    rf.fit(X_train_tfidf, y_train)
    
    end_time_step = time.time()
    print(f" Trained {rf.n_estimators} trees (step took {end_time_step - start_time_step:.2f}s)")
    
    rf.n_estimators += trees_per_step  # Add more trees for next step

end_time_total = time.time()
print(f"Random Forest training completed in {end_time_total - start_time_total:.2f} seconds")



In [ ]:
# -----------------------------
# 6. Evaluate
# -----------------------------
y_pred = rf.predict(X_test_tfidf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))